# SemiMind AI – Phase 1: AI4I Initial Analysis

> **Phase 1 | Notebook 2 of 3**  
> Dataset: AI4I 2020 Predictive Maintenance Dataset  
> Source: https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset

---

### Dataset Overview

The AI4I 2020 dataset simulates industrial equipment monitoring data for predictive maintenance research.  
Each row represents one measurement point with sensor readings and failure indicators.

**Columns discovered during Phase 1 inspection:**

| Column | Type | Description |
|--------|------|-------------|
| UDI | int | Unique identifier (1 – 10 000) |
| Product ID | str | Product serial number |
| Type | str | Machine quality variant (L / M / H) |
| Air temperature [K] | float | Air temperature in Kelvin |
| Process temperature [K] | float | Process temperature in Kelvin |
| Rotational speed [rpm] | int | Motor rotational speed |
| Torque [Nm] | float | Motor torque |
| Tool wear [min] | int | Cumulative tool wear in minutes |
| Machine failure | int | **Target** – 0 = no failure, 1 = failure |
| TWF | int | Tool Wear Failure indicator |
| HDF | int | Heat Dissipation Failure indicator |
| PWF | int | Power Failure indicator |
| OSF | int | Overstrain Failure indicator |
| RNF | int | Random Failure indicator |


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

from src.utils.config import load_config
from src.utils.logger import setup_logging
from src.ingestion.dataset_loader import load_ai4i_from_config

setup_logging(level='WARNING', log_to_file=False)
cfg = load_config()

df = load_ai4i_from_config(cfg)
print(f'AI4I loaded → shape: {df.shape}')

## First 5 and Last 5 Rows

In [ ]:
print('--- First 5 rows ---')
display(df.head())
print('--- Last 5 rows ---')
display(df.tail())

## Shape and Column Information

In [ ]:
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print()
print('Column data types:')
display(df.dtypes.to_frame(name='dtype'))

## Numerical Summary

In [ ]:
display(df.describe().round(3))

## Missing Value Inspection

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Missing values per column:')
display(mv_df)
print(f'\nTotal missing cells: {missing.sum()} out of {df.size:,}')

## Target Distribution

In [ ]:
target_vc = df['Machine failure'].value_counts().rename({0: 'No Failure (0)', 1: 'Failure (1)'})
print('Machine failure distribution:')
print(target_vc.to_string())
print(f'\nFailure rate: {df["Machine failure"].mean() * 100:.2f}%')
print(f'Imbalance ratio: {target_vc.max() / target_vc.min():.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
target_vc_plot = df['Machine failure'].value_counts()
axes[0].bar(['No Failure (0)', 'Failure (1)'], target_vc_plot.values, color=['steelblue', 'tomato'])
axes[0].set_title('Machine Failure – Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_vc_plot.values):
    axes[0].text(i, v + 30, str(v), ha='center')

# Pie chart
axes[1].pie(target_vc_plot.values, labels=['No Failure', 'Failure'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Machine Failure – Class Proportions')

plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'ai4i' / 'ai4i_target_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

## Sensor Feature Distributions

In [ ]:
numeric_cols = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')   # hide unused subplot
plt.suptitle('AI4I – Sensor Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'ai4i' / 'ai4i_sensor_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()

## Failure Type Distribution

In [ ]:
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_counts = df[failure_cols].sum()

fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.Set2.colors
ax.bar(failure_counts.index, failure_counts.values, color=colors[:5])
ax.set_title('Failure Type Counts (TWF / HDF / PWF / OSF / RNF)')
ax.set_ylabel('Count')
for i, (k, v) in enumerate(failure_counts.items()):
    ax.text(i, v + 1, str(v), ha='center')
plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'ai4i' / 'ai4i_failure_types.png'), dpi=120, bbox_inches='tight')
plt.show()

## Machine Type Distribution

In [ ]:
type_vc = df['Type'].value_counts()
print('Machine type distribution:')
print(type_vc.to_string())

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(type_vc.index, type_vc.values, color=['#4e79a7', '#f28e2b', '#e15759'])
ax.set_title('Machine Type Distribution (L / M / H)')
ax.set_ylabel('Count')
for i, v in enumerate(type_vc.values):
    ax.text(i, v + 30, str(v), ha='center')
plt.tight_layout()
plt.show()

## Initial Observations

**Findings from AI4I initial inspection (Phase 1):**

1. **No missing values** — All 10,000 rows have complete sensor readings.
2. **Class imbalance** — Machine failures are rare (~3.4%). The majority class significantly outnumbers the minority class.
3. **Multiple failure types** — HDF and OSF are the most common failure causes; RNF is nearly absent.
4. **Temperature features** — Air and process temperatures are tightly distributed (process temp ~ air temp + 10 K).
5. **Rotational speed** — Right-skewed; a small number of units operate at very high speeds.
6. **Machine type split** — L-type machines dominate (60%), followed by M (30%) and H (10%).
7. **Tool wear** — Uniformly distributed from 0 to ~250 minutes.

**Phase 2 priorities for this dataset:**
- Handle class imbalance during model training.
- Investigate correlation between failure types and sensor readings.
- Assess whether `UDI` and `Product ID` should be excluded from modelling.
